
# Telecom Customer Churn Prediction and Retention Optimization

## 02 - Data Cleaning and Validation

### Objective

Prepare a clean, consistent and validated customer
dataset for business analytics, statistical analysis,
machine learning and Power BI reporting.

### Tasks

- Load the original dataset.
- Standardize column names.
- Validate missing values.
- Investigate repeated records.
- Validate categorical and numerical variables.
- Generate unique record identifiers.
- Document data cleaning decisions.
- Export the cleaned dataset.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Raw Dataset

In [2]:
# Identify the project directory
project_root = Path.cwd()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

# Define raw dataset path
raw_data_path = (
    project_root / "data" / "raw" / "Customer Churn.csv"
)

# Load the original dataset
df_raw = pd.read_csv(raw_data_path)

# Create a working copy
df = df_raw.copy()

print("Raw dataset loaded successfully!")
print("Original dataset shape:", df_raw.shape)

Raw dataset loaded successfully!
Original dataset shape: (3150, 14)


## 2. Column Name Standardization

Convert column names into a consistent snake_case
format to improve readability and compatibility
with Python, SQL and Power BI.

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print("Standardized column names:")

for column in df.columns:
    print(column)

Standardized column names:
call_failure
complains
subscription_length
charge_amount
seconds_of_use
frequency_of_use
frequency_of_sms
distinct_called_numbers
age_group
tariff_plan
status
age
customer_value
churn


In [4]:
df = df.rename(columns={
    "complains": "has_complaint",
    "call_failure": "call_failures",
    "frequency_of_use": "call_frequency",
    "frequency_of_sms": "sms_frequency",
    "subscription_length": "subscription_months",
    "seconds_of_use": "call_duration_seconds"
})

df.head()

,call_failures,has_complaint,subscription_months,charge_amount,call_duration_seconds,call_frequency,sms_frequency,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value,churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [5]:
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])
print("Column names are unique:", df.columns.is_unique)
print("Total missing values:", df.isnull().sum().sum())

df.info()

Total records: 3150
Total columns: 14
Column names are unique: True
Total missing values: 0
<class 'pandas.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   call_failures            3150 non-null   int64  
 1   has_complaint            3150 non-null   int64  
 2   subscription_months      3150 non-null   int64  
 3   charge_amount            3150 non-null   int64  
 4   call_duration_seconds    3150 non-null   int64  
 5   call_frequency           3150 non-null   int64  
 6   sms_frequency            3150 non-null   int64  
 7   distinct_called_numbers  3150 non-null   int64  
 8   age_group                3150 non-null   int64  
 9   tariff_plan              3150 non-null   int64  
 10  status                   3150 non-null   int64  
 11  age                      3150 non-null   int64  
 12  customer_value           3150 non-null   float64
 1

## 3. Duplicate Record Handling

Investigate repeated customer records and preserve
observations where individual customer identity
cannot be verified.

In [6]:
duplicate_count = df.duplicated().sum()

print("Total records:", len(df))
print("Repeated records:", duplicate_count)
print("Unique record combinations:", len(df.drop_duplicates()))

print(
    "Repeated record percentage:",
    round(duplicate_count / len(df) * 100, 2),
    "%"
)

Total records: 3150
Repeated records: 300
Unique record combinations: 2850
Repeated record percentage: 9.52 %


In [7]:
df["is_duplicate_record"] = (
    df.duplicated(keep=False).astype(int)
)

print(df["is_duplicate_record"].value_counts())

is_duplicate_record
0    2685
1     465
Name: count, dtype: int64


## 4. Categorical Variable Validation

Validate the observed categories against the
expected values and identify invalid records.

In [8]:
valid_categories = {
    "has_complaint": [0, 1],
    "age_group": [1, 2, 3, 4, 5],
    "tariff_plan": [1, 2],
    "status": [1, 2],
    "churn": [0, 1]
}

validation_results = []

for column, valid_values in valid_categories.items():

    invalid_count = (~df[column].isin(valid_values)).sum()

    validation_results.append({
        "Column": column,
        "Expected Values": valid_values,
        "Invalid Records": invalid_count,
        "Result": "PASS" if invalid_count == 0 else "FAIL"
    })

category_validation = pd.DataFrame(validation_results)

category_validation

,Column,Expected Values,Invalid Records,Result
0,has_complaint,"[0, 1]",0,PASS
1,age_group,"[1, 2, 3, 4, 5]",0,PASS
2,tariff_plan,"[1, 2]",0,PASS
3,status,"[1, 2]",0,PASS
4,churn,"[0, 1]",0,PASS


## 5. Charge Amount Validation

Investigate the difference between the documented
charge categories and the observed dataset values.

In [9]:
print("Minimum charge category:",
      df["charge_amount"].min())

print("Maximum charge category:",
      df["charge_amount"].max())

print("Records with category 10:",
      (df["charge_amount"] == 10).sum())

print("Records outside the observed 0-10 range:",
      (~df["charge_amount"].between(0, 10)).sum())

df["charge_amount"].value_counts().sort_index()

Minimum charge category: 0
Maximum charge category: 10
Records with category 10: 7
Records outside the observed 0-10 range: 0


charge_amount
0     1768
1      617
2      395
3      199
4       76
5       30
6       11
7       14
8       19
9       14
10       7
Name: count, dtype: int64

## 6. Numerical Variable Validation

Check numerical customer attributes for negative
values and other potential data quality issues.

In [10]:
numerical_columns = [
    "call_failures",
    "subscription_months",
    "call_duration_seconds",
    "call_frequency",
    "sms_frequency",
    "distinct_called_numbers",
    "age",
    "customer_value"
]

numerical_validation = pd.DataFrame({
    "Minimum": df[numerical_columns].min(),
    "Maximum": df[numerical_columns].max(),
    "Negative Values": (
        df[numerical_columns] < 0
    ).sum(),
    "Missing Values": (
        df[numerical_columns].isnull().sum()
    )
})

numerical_validation

,Minimum,Maximum,Negative Values,Missing Values
call_failures,0.0,36.00,0,0
subscription_months,3.0,47.00,0,0
call_duration_seconds,0.0,17090.00,0,0
call_frequency,0.0,255.00,0,0
sms_frequency,0.0,522.00,0,0
distinct_called_numbers,0.0,97.00,0,0
age,15.0,55.00,0,0
customer_value,0.0,2165.28,0,0


## 7. Record Identifier Generation

Generate a unique identifier for each observation
to support database integration and record tracking.

The generated identifier represents a dataset record,
not a verified real-world customer identity.

In [11]:
df.insert(
    0,
    "record_id",
    [f"CUST{i:05d}" for i in range(1, len(df) + 1)]
)

df[["record_id", "churn"]].head()

,record_id,churn
0,CUST00001,0
1,CUST00002,0
2,CUST00003,0
3,CUST00004,0
4,CUST00005,0


In [12]:
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])

print(
    "Unique record IDs:",
    df["record_id"].nunique()
)

print(
    "Record IDs are unique:",
    df["record_id"].is_unique
)

print(
    "Total missing values:",
    df.isnull().sum().sum()
)

df.head()

Total records: 3150
Total columns: 16
Unique record IDs: 3150
Record IDs are unique: True
Total missing values: 0


,record_id,call_failures,has_complaint,subscription_months,charge_amount,call_duration_seconds,call_frequency,sms_frequency,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value,churn,is_duplicate_record
0,CUST00001,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0,1
1,CUST00002,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0,1
2,CUST00003,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0,1
3,CUST00004,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0,1
4,CUST00005,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0,1


## 8. Automated Data Quality Checks

Perform automated validation checks to ensure that
the cleaned dataset satisfies the required data
quality conditions before exporting.

In [13]:
validation_checks = {
    "Original record count preserved":
        len(df) == len(df_raw),

    "Expected number of columns":
        df.shape[1] == 16,

    "Unique column names":
        df.columns.is_unique,

    "No missing values":
        df.isnull().sum().sum() == 0,

    "Unique record identifiers":
        df["record_id"].is_unique,

    "Valid categorical values":
        all(
            df[col].isin(values).all()
            for col, values in valid_categories.items()
        ),

    "Valid observed charge categories":
        df["charge_amount"].isin(range(11)).all(),

    "Valid numerical values":
        (df[numerical_columns] >= 0).all().all(),

    "Finite numerical values":
        np.isfinite(df[numerical_columns]).all().all(),

    "Valid duplicate indicator":
        df["is_duplicate_record"].isin([0, 1]).all()
}

validation_summary = pd.DataFrame({
    "Validation Check": list(validation_checks.keys()),
    "Result": [
        "PASS" if result else "FAIL"
        for result in validation_checks.values()
    ]
})

validation_summary

,Validation Check,Result
0,Original record count preserved,PASS
1,Expected number of columns,PASS
2,Unique column names,PASS
3,No missing values,PASS
4,Unique record identifiers,PASS
5,Valid categorical values,PASS
6,Valid observed charge categories,PASS
7,Valid numerical values,PASS
8,Finite numerical values,PASS
9,Valid duplicate indicator,PASS


## 9. Raw and Cleaned Dataset Comparison

Compare the original and cleaned datasets to verify
the changes performed during data preparation.

In [14]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Columns",
        "Missing Values",
        "Repeated Source Records",
        "Unique Record IDs",
        "Churn Rate (%)"
    ],

    "Raw Dataset": [
        len(df_raw),
        df_raw.shape[1],
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        "Not Available",
        round(df_raw["Churn"].mean() * 100, 2)
    ],

    "Cleaned Dataset": [
        len(df),
        df.shape[1],
        df.isnull().sum().sum(),
        df.drop(
            columns=["record_id", "is_duplicate_record"]
        ).duplicated().sum(),
        df["record_id"].nunique(),
        round(df["churn"].mean() * 100, 2)
    ]
})

cleaning_summary

,Metric,Raw Dataset,Cleaned Dataset
0,Total Records,3150,3150.00
1,Total Columns,14,16.00
2,Missing Values,0,0.00
3,Repeated Source Records,300,300.00
4,Unique Record IDs,Not Available,3150.00
5,Churn Rate (%),15.71,15.71


## 10. Export Cleaned Dataset

Export the validated customer dataset for PostgreSQL,
statistical analysis, machine learning and Power BI.

In [15]:
# Stop the export if any validation fails
assert all(validation_checks.values()), (
    "Data validation failed. Review the results."
)

# Define output directory
output_directory = project_root / "data" / "processed"

# Ensure the directory exists
output_directory.mkdir(parents=True, exist_ok=True)

# Define output file
cleaned_data_path = (
    output_directory / "telecom_churn_cleaned.csv"
)

# Export cleaned dataset
df.to_csv(
    cleaned_data_path,
    index=False,
    encoding="utf-8"
)

print("Cleaned dataset exported successfully!")
print("File location:", cleaned_data_path)

Cleaned dataset exported successfully!
File location: c:\Users\kavin\OneDrive - University of Kelaniya\Desktop\telecom-churn-retention-intelligence\data\processed\telecom_churn_cleaned.csv


In [16]:
# Reload the exported dataset
df_cleaned = pd.read_csv(cleaned_data_path)

# Compare the exported data with the working dataset
pd.testing.assert_frame_equal(
    df_cleaned,
    df,
    check_dtype=False
)

print("Exported dataset verified successfully!")
print("Total records:", df_cleaned.shape[0])
print("Total columns:", df_cleaned.shape[1])
print("Missing values:", df_cleaned.isnull().sum().sum())
print("Unique IDs:", df_cleaned["record_id"].nunique())

df_cleaned.head()

Exported dataset verified successfully!
Total records: 3150
Total columns: 16
Missing values: 0
Unique IDs: 3150


,record_id,call_failures,has_complaint,subscription_months,charge_amount,call_duration_seconds,call_frequency,sms_frequency,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value,churn,is_duplicate_record
0,CUST00001,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0,1
1,CUST00002,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0,1
2,CUST00003,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0,1
3,CUST00004,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0,1
4,CUST00005,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0,1


In [17]:
report_directory = project_root / "reports"

report_directory.mkdir(
    parents=True,
    exist_ok=True
)

validation_summary.to_csv(
    report_directory / "data_validation_report.csv",
    index=False
)

cleaning_summary.to_csv(
    report_directory / "data_cleaning_summary.csv",
    index=False
)

print("Data quality reports exported successfully!")

Data quality reports exported successfully!



## 11. Data Cleaning Summary and Conclusion

### Completed Cleaning Tasks

- Standardized column names using snake_case.
- Renamed selected columns for improved readability.
- Verified that no missing values exist.
- Investigated repeated customer records.
- Created a duplicate record indicator.
- Validated numerical and categorical variables.
- Generated unique record identifiers.
- Implemented automated data quality checks.
- Exported and verified the cleaned dataset.
- Generated data quality reports.

### Final Dataset

- Total records: 3,150
- Total columns: 16
- Missing values: 0
- Unique record identifiers: 3,150
- Customer churn rate: 15.71%

### Data Cleaning Decisions

Repeated records were retained because the original
dataset does not provide verified customer identifiers.

Charge category 10 was retained because there is
insufficient evidence to justify modifying it.

The generated record identifier and duplicate indicator
are intended for auditing and record tracking and will
not be used as predictive model features.

Identical predictor records will be grouped appropriately
during machine learning dataset splitting to reduce
the risk of data leakage.

### Conclusion

The customer dataset has been standardized, validated
and prepared for database integration, business analysis,
statistical investigation and predictive modelling.
